# Counterfeit & Illegal Medicines in Indonesia (2015–2026)
### Interactive Map using Folium

This notebook builds an interactive map of BPOM enforcement actions against counterfeit medicines in Indonesia.

Based on: *Kesmas: National Public Health Journal* (2017). Counterfeit Medicines in Socioeconomic Perspective, 11(4), 153. https://doi.org/10.21109/kesmas.v11i4.1440

Run each cell in order using **Shift + Enter**.

In [ ]:
# Cell 1 - install and import libraries
# if folium is not installed yet, uncomment the line below and run it first
# !pip install folium

import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MarkerCluster
import warnings
warnings.filterwarnings('ignore')

print('pandas:', pd.__version__)
print('folium:', folium.__version__)
print('ready!')

In [ ]:
# Cell 2 - load the dataset
df = pd.read_csv('../data/indonesia_counterfeit_medicines_2015_2026.csv')

# remove rows where coordinates are missing
df = df.dropna(subset=['latitude', 'longitude'])

# make sure lat/lng are numbers
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['year']      = pd.to_numeric(df['year'],      errors='coerce')

# fill empty text fields
text_cols = ['city', 'province', 'drug_category', 'brands_seized',
             'action_taken', 'channel', 'notes', 'source_url']
df[text_cols] = df[text_cols].fillna('-')

print(f'{len(df)} cases loaded')
df[['year', 'city', 'province', 'drug_category', 'channel']].head()

In [ ]:
# Cell 3 - create the base map
m = folium.Map(
    location=[-2.5, 118.0],
    zoom_start=5,
    tiles='CartoDB positron'
)

m

In [ ]:
# Cell 4 - add markers
# colour is based on drug category, icon is based on whether it was sold online or physically

def get_colour(category):
    cat = str(category).lower()
    if 'vaccine' in cat:                            return 'red'
    elif 'erectile' in cat or 'slimming' in cat:   return 'purple'
    elif 'traditional' in cat or 'jamu' in cat:    return 'green'
    elif 'syrup' in cat:                            return 'orange'
    elif 'psychotropic' in cat or 'pcc' in cat:    return 'darkred'
    elif 'supplement' in cat:                       return 'cadetblue'
    elif 'stem cell' in cat or 'secretome' in cat: return 'pink'
    else:                                           return 'blue'

def get_icon(channel):
    return 'cloud' if 'online' in str(channel).lower() else 'home'

cluster = MarkerCluster(name='All cases').add_to(m)

for _, row in df.iterrows():
    try:
        val = f"Rp {int(float(row['estimated_value_idr'])):,}"
    except:
        val = 'not reported'

    popup_html = f"""
    <div style="font-family:Arial; font-size:13px; width:280px;">
        <b>Case {int(row['case_id'])} &mdash; {int(row['year'])}</b><br>
        <b>Location:</b> {row['city']}, {row['province']}<br>
        <b>Drug type:</b> {row['drug_category']}<br>
        <b>Channel:</b> {row['channel']}<br>
        <b>Value seized:</b> {val}<br>
        <b>Action:</b> {row['action_taken']}<br>
        <hr style="margin:5px 0;">
        <small>{str(row['notes'])[:120]}...</small><br>
        <a href="{row['source_url']}" target="_blank">Source</a>
    </div>
    """

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{int(row['year'])} | {row['city']} | {row['drug_category']}",
        icon=folium.Icon(
            color=get_colour(row['drug_category']),
            icon=get_icon(row['channel']),
            prefix='glyphicon'
        )
    ).add_to(cluster)

print(f'{len(df)} markers added')
m

In [ ]:
# Cell 5 - add heatmap
# weight is based on log of value seized so high-value cases glow brighter

def get_weight(x):
    try:
        v = float(x)
        return np.log1p(v) / 20 if v > 0 else 1
    except:
        return 1

df['weight'] = df['estimated_value_idr'].apply(get_weight)

heat_data = [[row['latitude'], row['longitude'], row['weight']]
             for _, row in df.iterrows()]

HeatMap(
    heat_data,
    name='Heatmap',
    radius=40,
    blur=25,
    max_zoom=8,
    gradient={'0.2': 'blue', '0.5': 'orange', '1.0': 'red'}
).add_to(m)

print('heatmap added')
m

In [ ]:
# Cell 6 - separate layers for online vs physical cases

online_layer   = folium.FeatureGroup(name='Online cases',   show=False)
physical_layer = folium.FeatureGroup(name='Physical raids', show=False)

for _, row in df.iterrows():
    channel = str(row['channel']).lower()
    colour  = '#e74c3c' if 'online' in channel else '#27ae60'

    circle = folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=9,
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.7,
        tooltip=f"{int(row['year'])} | {row['city']}"
    )

    if 'online' in channel:
        circle.add_to(online_layer)
    else:
        circle.add_to(physical_layer)

online_layer.add_to(m)
physical_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

print('layers added')
m

In [ ]:
# Cell 7 - add legend and title

legend_html = """
<div style="position:fixed; bottom:40px; left:40px; z-index:1000;
            background:white; border:1px solid #ccc; border-radius:8px;
            padding:12px 16px; font-family:Arial; font-size:12px;">
  <b>Indonesia Counterfeit Medicines 2015&ndash;2026</b>
  <hr style="margin:6px 0;">
  <b>Marker colour:</b><br>
  <span style="color:red;">&#9679;</span> Counterfeit vaccines<br>
  <span style="color:purple;">&#9679;</span> ED / slimming drugs<br>
  <span style="color:green;">&#9679;</span> Illegal traditional medicines<br>
  <span style="color:orange;">&#9679;</span> Contaminated syrups<br>
  <span style="color:#e91e63;">&#9679;</span> Stem cell / biologics<br>
  <span style="color:blue;">&#9679;</span> Multiple / online<br>
  <hr style="margin:6px 0;">
  <b>Icon:</b> &#9729; Online &nbsp; &#8962; Physical<br>
  <hr style="margin:6px 0;">
  <small>Source: BPOM press releases 2015&ndash;2026</small>
</div>
"""

title_html = """
<div style="position:fixed; top:16px; left:50%; transform:translateX(-50%);
            z-index:1000; background:rgba(255,255,255,0.92);
            border:1.5px solid #c0392b; border-radius:8px;
            padding:7px 20px; text-align:center;">
  <b style="font-size:14px; color:#c0392b;">Counterfeit & Illegal Medicines &mdash; Indonesia 2015&ndash;2026</b><br>
  <small>BPOM Enforcement Data | 28 cases</small>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))
m.get_root().html.add_child(folium.Element(title_html))

print('legend and title added')
m

In [ ]:
# Cell 8 - save the final map
import os
os.makedirs('../outputs', exist_ok=True)

m.save('../outputs/indonesia_counterfeit_map.html')
print('map saved to outputs/indonesia_counterfeit_map.html')
print('open that file in any browser to view the full interactive map')